In [1]:
import numpy as np
import re
import random
import requests
from collections import Counter
from tqdm.auto import tqdm
import time

urls = [("Война и мир", "https://raw.githubusercontent.com/sismetanin/word2vec-tsne/master/data/War%20and%20Peace%20by%20Leo%20Tolstoy%20(ru).txt")]

raw_text = ""
for name, url in urls:
    try:
        print(f"  Загрузка: {name}...")
        resp = requests.get(url, timeout=30)
        for enc in ['utf-8', 'cp1251', 'koi8-r']:
            try:
                text = resp.content.decode(enc)
                if sum(1 for c in text[:1000] if 'а' <= c <= 'я' or 'А' <= c <= 'Я') > 50:
                    raw_text += " " + text
                    print(f"    {name}: {len(text)} символов, кодировка {enc}")
                    break
            except:
                continue
    except Exception as e:
        print(f"    {name}: {e}")

print(f"Загружено символов: {len(raw_text)}")

text_clean = re.sub(r'[^а-яёА-ЯЁ\s]', ' ', raw_text)
text_clean = text_clean.lower()
words_all = text_clean.split()
words_all = [w for w in words_all if len(w) > 2]

/Users/yoonzky/venvs/ai/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Загрузка: Война и мир...
    Война и мир: 3114987 символов, кодировка cp1251
Загружено символов: 3114988


In [2]:
MIN_FREQ = 5
freq = Counter(words_all)
vocab_words = sorted([w for w, c in freq.items() if c >= MIN_FREQ])

word2idx = {w: i for i, w in enumerate(vocab_words)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(word2idx)
print(f"Размер словаря (freq >= {MIN_FREQ}): {V}")

corpus = np.array([word2idx[w] for w in words_all if w in word2idx], dtype=np.int32)
print(f"Длина корпуса: {len(corpus)}")

Размер словаря (freq >= 5): 9400
Длина корпуса: 291366


In [3]:
freqs = np.zeros(V, dtype=np.float64)
for idx in corpus:
    freqs[idx] += 1

freqs_pow = freqs ** 0.75
freqs_pow /= freqs_pow.sum()

freq_probs = freqs_pow
"""
def get_negatives_batch(n, exclude_idx):
    negs = []
    while len(negs) < n:
        idx = np.random.choice(V, p=freq_probs)
        if idx != exclude_idx:
            negs.append(idx)
    return np.array(negs, dtype=np.int32)
"""
TABLE_SIZE = 10000000
neg_table = np.zeros(TABLE_SIZE, dtype=np.int32)
kymsum= np.cumsum(freq_probs)
j = 0
for i in range(TABLE_SIZE):
    while j < V - 1 and kymsum[j] < (i + 0.5) / TABLE_SIZE:
        j += 1
    neg_table[i] = j

def get_negatives_batch(n, exclude_idx):
    negs = np.empty(n, dtype=np.int32)
    count = 0
    while count < n:
        batch = neg_table[np.random.randint(0, TABLE_SIZE, size=n - count)]
        mask = batch != exclude_idx
        good = batch[mask]
        end = min(count + len(good), n)
        negs[count:end] = good[:end - count]
        count = end
    return negs

In [4]:
C_POS = 3
C_NEG = 10
EMB_DIMS = [300]

In [5]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -8, 8)))

def train_word2vec(emb_dim, epochs=3, lr=0.05):
    print(f"Обучение: dim={emb_dim}, epochs={epochs}, lr={lr}")

    scale = 0.5 / emb_dim
    W_target  = np.random.uniform(-scale, scale, (V, emb_dim)).astype(np.float32)
    W_context = np.random.uniform(-scale, scale, (V, emb_dim)).astype(np.float32)

    total_steps = epochs * len(corpus)
    step = 0

    for epoch in range(epochs):
        pbar = tqdm(range(len(corpus)), desc=f"Epoch {epoch+1}/{epochs}")

        for pos in pbar:

            current_lr = lr * max(1.0 - step / total_steps, 1e-4)
            step += 1

            center_idx = corpus[pos]

            start = max(0, pos - C_POS)
            end = min(len(corpus), pos + C_POS + 1)

            for ctx_pos in range(start, end):
                if ctx_pos == pos:
                    continue
                context_idx = corpus[ctx_pos]

                w = W_target[center_idx].copy()

                c_pos = W_context[context_idx] # вектор контекстного слова
                dot_pos = np.dot(c_pos, w)
                sig_pos = sigmoid(dot_pos)

                W_context[context_idx] -= current_lr * (sig_pos - 1.0) * w # c_pos^t+1

                # dL/dw первая часть
                grad_w = (sig_pos - 1.0) * c_pos

                neg_indices = get_negatives_batch(C_NEG, context_idx) # массив из индексов случайных слов
                C_neg_mat = W_context[neg_indices] # 10 векторов этих слов, 10*dim
                dots_neg = C_neg_mat @ w # 10 скалярных произведение: c_negi * w
                sig_neg = sigmoid(dots_neg) # 10 сигмоид sigm(c_negi * w)

                # dL/dc_negi = sigm(c_negi*w) * w
                grad_C_neg = sig_neg[:, np.newaxis] * w[np.newaxis, :] # матрица 10*dim где i строка sigm(c_negi*w) * w

                """
                dim = 3
                sig_neg = [0.52, 0.48]  2 негативных слова
                w = [0.1, 0.2, 0.3]

                sig_neg[:, np.newaxis]:
                [0.52, 0.48]  ->  [[0.52],
                                    0.48]]

                w[np.newaxis, :]:
                [0.1, 0.2, 0.3]  ->  [[0.1, 0.2, 0.3]]

                умножение их:
                [[0.52],     *   [[0.1, 0.2, 0.3]]
                [0.48]]
                                  =
                      [[0.52*0.1, 0.52*0.2, 0.52*0.3],
                      [0.48*0.1, 0.48*0.2, 0.48*0.3]]
                """

                W_context[neg_indices] -= current_lr * grad_C_neg # c_neg^t+1

                # dL/dw первая часть + summa neg
                grad_w += (sig_neg[:, np.newaxis] * C_neg_mat).sum(axis=0)

                # w^t+1
                W_target[center_idx] -= current_lr * grad_w

        print(f"Epoch {epoch+1} done")

    return W_target, W_context

models = {}
for dim in EMB_DIMS:
    if dim <= 100:
        ep = 3
    elif dim <= 500:
        ep = 3
    else:
        ep = 3
    W_t, W_c = train_word2vec(dim, epochs=ep, lr=0.05)
    models[dim] = W_t

Обучение: dim=300, epochs=3, lr=0.05


Epoch 1/3: 100%|██████████| 291366/291366 [00:35<00:00, 8216.39it/s]


Epoch 1 done


Epoch 2/3: 100%|██████████| 291366/291366 [00:35<00:00, 8295.74it/s]


Epoch 2 done


Epoch 3/3: 100%|██████████| 291366/291366 [00:35<00:00, 8282.46it/s]

Epoch 3 done


In [6]:
def find_nearest(word, W, top_k=10, metric='mse'):
    idx = word2idx[word]
    vec = W[idx]

    if metric == 'mse':
        diffs = W - vec[np.newaxis, :]
        distances = np.mean(diffs ** 2, axis=1)
        distances[idx] = np.inf
        top_indices = np.argsort(distances)[:top_k]
        return [(idx2word[i], float(distances[i])) for i in top_indices]

In [7]:
test_words = ["война", "мир", "мужчина", "александр", "девушка", "любовь", "солдат", "наполеон", "смерть", "армия", "император"]

for dim in EMB_DIMS:
    W = models[dim]

    print(f"\ndim={dim} MSE")
    for word in test_words:
        neighbors = find_nearest(word, W, top_k=5, metric='mse')
        res = ", ".join([f"{w} ({s:.6f})" for w, s in neighbors])
        print(f"  {word:12s} -> {res}")


dim=300 MSE
  война        -> сила (0.072118), история (0.095159), сущность (0.098407), причина (0.104350), находится (0.112920)
  мир          -> тела (0.058009), россию (0.063577), словах (0.066068), ужас (0.067021), приказанию (0.067863)
  мужчина      -> одет (0.042923), длинный (0.043788), худой (0.045321), шее (0.050106), черный (0.050352)
  александр    -> слышали (0.076054), приказ (0.080751), порядок (0.083828), характер (0.086487), здравствует (0.086519)
  девушка      -> осталась (0.094202), женщина (0.103855), ваша (0.104443), москва (0.105586), обратилась (0.105929)
  любовь       -> отношении (0.111959), сущность (0.143054), душу (0.151863), желала (0.152766), всякое (0.154906)
  солдат       -> другим (0.167322), пленных (0.168197), офицеров (0.172911), французский (0.174180), новый (0.174597)
  наполеон     -> случай (0.155003), сознание (0.155856), хотя (0.158445), состояние (0.165438), морель (0.166140)
  смерть       -> присутствие (0.079038), воля (0.081749), страш

In [8]:
print("Несвязные слова")

diff_pairs = [
    ("война", "войны"),
    ("солдат", "девушка"),
    ("князь", "дерево"),
    ("москва", "лошадь"),
    ("армия", "цвет"),
    ("наполеон", "девушка"),
]


for dim in EMB_DIMS:
    W = models[dim]
    norms = np.linalg.norm(W, axis=1, keepdims=True) + 1e-10
    W_n = W / norms

    print(f"\ndim={dim}")
    print("Пары:")
    for w1, w2 in diff_pairs:
        if w1 in word2idx and w2 in word2idx:
            dot = np.dot(W_n[word2idx[w1]], W_n[word2idx[w2]])
            print(f"    {w1} * {w2} = {dot:.4f}")


Несвязные слова

dim=300
Пары:
    война * войны = 0.6720
    солдат * девушка = 0.0743
    князь * дерево = 0.0363
    москва * лошадь = 0.4360
    армия * цвет = 0.1574
    наполеон * девушка = 0.1413


LB2

In [9]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import re

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Эмбеддинги из лаб. 1
EMB_DIM = 300
W_emb = models[EMB_DIM]

# Переводим эмбеддинги в тензор на GPU
emb_tensor = torch.tensor(W_emb, dtype=torch.float32, device=device)

# Параметры
HIDDEN_SIZE = 512
EPOCHS = 25
LR = 0.001
BATCH_SIZE = 128
MIN_LEN = 3   # минимальная длина предложения (слов)
MAX_LEN = 30  # максимальная длина предложения

Device: cpu


In [10]:
# 1. Разбиение текста на кусочные предложения по запятым, тире, точкам

print("Разбиение текста на кусочные предложения")

# Разбиваем исходный текст по знакам препинания
chunks = re.split(r'[,.\-;:!?\n]+', raw_text.lower())

sentences = []
for chunk in chunks:
    # Очищаем только кириллица
    clean = re.sub(r'[^а-яёа-яё\s]', ' ', chunk)
    words = clean.split()
    # Удаляем короткие слова (<= 2 буквы) и оставляем только слова из словаря
    words = [w for w in words if len(w) > 2 and w in word2idx]
    if MIN_LEN <= len(words) <= MAX_LEN:
        sentences.append([word2idx[w] for w in words])

print(f"Всего кусочных предложений: {len(sentences)}")
print(f"Средняя длина: {np.mean([len(s) for s in sentences]):.1f} слов")
print(f"Пример: {' '.join([idx2word[i] for i in sentences[0]])}")

Разбиение текста на кусочные предложения
Всего кусочных предложений: 48200
Средняя длина: 4.5 слов
Пример: книгу вошли первый второй романа война мир одного самых века


In [11]:
# 2. Обучающие выборки
# Вход: предложение без последнего слова
# Цель: предложение без первого слова (сдвиг на 1)
# -> на каждом шаге предсказываем следующее слово

print("Построение обучающих выборок...")

# Разбиение train/test (90/10)
np.random.seed(42)
perm = np.random.permutation(len(sentences))
split = int(len(sentences) * 0.9)
train_sents = [sentences[i] for i in perm[:split]]
test_sents = [sentences[i] for i in perm[split:]]

print(f"Train предложений: {len(train_sents)}, Test предложений: {len(test_sents)}")

Построение обучающих выборок...
Train предложений: 43380, Test предложений: 4820


In [12]:
# 3. Модель: Encoder-Decoder LSTM

class LSTMLanguageModel(nn.Module):
    def __init__(self, emb_dim, hidden_size, vocab_size, emb_weights, dropout=0.3):
        super().__init__()
        # Слой эмбеддингов из word2vec
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.embedding.weight = nn.Parameter(torch.tensor(emb_weights, dtype=torch.float32))
        self.embedding.weight.requires_grad = False

        self.dropout = nn.Dropout(dropout)

        # Энкодер LSTM — накапливает скрытое состояние
        self.encoder = nn.LSTM(emb_dim, hidden_size, batch_first=True, dropout=0)

        # Декодер LSTM — принимает скрытое состояние энкодера и снова слова предложения
        self.decoder = nn.LSTM(emb_dim, hidden_size, batch_first=True, dropout=0)

        # Выходной слой softmax по всему словарю
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        emb = self.dropout(emb)

        # Энкодер проходит по всему предложению, накапливает состояние
        enc_out, (h_enc, c_enc) = self.encoder(emb)

        # Декодер принимает скрытое состояние энкодера и снова те же слова
        dec_out, _ = self.decoder(emb, (h_enc, c_enc))
        dec_out = self.dropout(dec_out)

        # Выходной слой на каждом шаге предсказываем следующее слово
        logits = self.fc(dec_out)

        return logits

W_emb_copy = W_emb.copy()
model = LSTMLanguageModel(EMB_DIM, HIDDEN_SIZE, V, W_emb_copy).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f"\nАрхитектура:")
print(f"Embedding: {V} x {EMB_DIM} (из word2vec)")
print(f"Encoder LSTM: input={EMB_DIM}, hidden={HIDDEN_SIZE}")
print(f"Decoder LSTM: input={EMB_DIM}, hidden={HIDDEN_SIZE}")
print(f"Output: Linear({HIDDEN_SIZE}, {V}) + Softmax")


Архитектура:
Embedding: 9400 x 300 (из word2vec)
Encoder LSTM: input=300, hidden=512
Decoder LSTM: input=300, hidden=512
Output: Linear(512, 9400) + Softmax


In [13]:
# 4. Функции для батчей

def collate_fn(batch):
    """
    Для каждого предложения [w1, w2, w3, w4, w5]:
      input  = [w1, w2, w3, w4]
      target = [w2, w3, w4, w5]
    """
    inputs = []
    targets = []
    lengths = []

    for sent in batch:
        inp = torch.tensor(sent[:-1], dtype=torch.long) # без последнего
        tgt = torch.tensor(sent[1:], dtype=torch.long) # без первого
        inputs.append(inp)
        targets.append(tgt)
        lengths.append(len(inp))

    # Паддинг до одной длины
    inputs_padded = pad_sequence(inputs, batch_first=True, padding_value=0)
    targets_padded = pad_sequence(targets, batch_first=True, padding_value=-1) # -1 = ignore
    lengths = torch.tensor(lengths, dtype=torch.long)

    return inputs_padded.to(device), targets_padded.to(device), lengths.to(device)

train_loader = DataLoader(train_sents, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_sents, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [14]:
# 5. Обучение

print(f"\nОбучение: epochs={EPOCHS}, lr={LR}, batch_size={BATCH_SIZE}")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for inputs, targets, lengths in pbar:
        optimizer.zero_grad()

        logits = model(inputs, lengths)  # (batch, seq_len, V)

        # Reshape для cross-entropy (batch*seq_len, V) vs (batch*seq_len,)
        logits_flat = logits.reshape(-1, V)
        targets_flat = targets.reshape(-1)

        loss = criterion(logits_flat, targets_flat)
        loss.backward()
        optimizer.step()

        # Считаем accuracy
        mask = targets_flat != -1
        preds = logits_flat.argmax(dim=1)
        correct += (preds[mask] == targets_flat[mask]).sum().item()
        total += mask.sum().item()
        total_loss += loss.item() * mask.sum().item()

        if total > 0 and total % 10000 < BATCH_SIZE * 10:
            pbar.set_postfix({"loss": f"{total_loss/total:.3f}", "acc": f"{correct/total:.2%}"})

    train_acc = correct / total
    train_loss = total_loss / total
    print(f"  Epoch {epoch+1}: loss={train_loss:.4f}, train_acc={train_acc:.2%}")



Обучение: epochs=25, lr=0.001, batch_size=128


Epoch 1/25: 100%|██████████| 339/339 [00:28<00:00, 12.01it/s, loss=7.646, acc=4.14%]


  Epoch 1: loss=7.6444, train_acc=4.16%


Epoch 2/25: 100%|██████████| 339/339 [00:29<00:00, 11.60it/s, loss=6.136, acc=15.19%]


  Epoch 2: loss=6.1348, train_acc=15.21%


...

  Epoch 23: loss=0.7588, train_acc=83.03%


Epoch 24/25: 100%|██████████| 339/339 [00:29<00:00, 11.36it/s, loss=0.722, acc=83.63%]


  Epoch 24: loss=0.7222, train_acc=83.63%


Epoch 25/25: 100%|██████████| 339/339 [00:30<00:00, 11.25it/s, loss=0.700, acc=83.93%]

  Epoch 25: loss=0.7006, train_acc=83.92%


In [15]:
# 6. Оценка на тестовой выборке: perplexity и accuracy
print("\nОценка на тестовой выборке:")

model.eval()
test_loss = 0.0
test_correct = 0
test_total = 0

with torch.no_grad():
    for inputs, targets, lengths in test_loader:
        logits = model(inputs, lengths)
        logits_flat = logits.reshape(-1, V)
        targets_flat = targets.reshape(-1)

        loss = criterion(logits_flat, targets_flat)
        mask = targets_flat != -1
        preds = logits_flat.argmax(dim=1)
        test_correct += (preds[mask] == targets_flat[mask]).sum().item()
        test_total += mask.sum().item()
        test_loss += loss.item() * mask.sum().item()

test_accuracy = test_correct / test_total
avg_test_loss = test_loss / test_total
perplexity = np.exp(avg_test_loss)

print(f"  Test accuracy: {test_accuracy:.2%}")
print(f"  Test perplexity: {perplexity:.2f}")



Оценка на тестовой выборке:
  Test accuracy: 48.46%
  Test perplexity: 75.33


In [16]:
# 7. Примеры предсказаний на тестовых предложениях
print("\nПримеры предсказаний на тестовых предложениях:")

model.eval()
shown_T = 0
shown_F = 0
with torch.no_grad():
    for inputs, targets, lengths in test_loader:
        logits = model(inputs, lengths)
        preds = logits.argmax(dim=2)  # (batch, seq_len)

        for i in range(len(lengths)):
            if shown_T >= 20 and shown_F >= 20:
                break
            L = lengths[i].item()

            input_words = [idx2word[inputs[i, j].item()] for j in range(L)]
            target_words = [idx2word[targets[i, j].item()] for j in range(L)]
            pred_words = [idx2word[preds[i, j].item()] for j in range(L)]

            # Показываем последнее слово
            true_last = target_words[-1]
            pred_last = pred_words[-1]
            mark = "T" if true_last == pred_last else "F"
            if mark == "T" and shown_T <= 20:
              context = ' '.join(input_words)
              print(f"  {context} -> предсказано: {pred_last}, правильно: {true_last} {mark}")
              shown_T += 1
            if mark == "F" and shown_F <= 20:
              context = ' '.join(input_words)
              print(f"  {context} -> предсказано: {pred_last}, правильно: {true_last} {mark}")
              shown_F += 1

        if shown_T >= 20 and shown_F >= 20:
            break


Примеры предсказаний на тестовых предложениях:
  они эти -> предсказано: находились, правильно: странные F
  что наташа сидела все утро окна -> предсказано: кровати, правильно: гостиной F
  дверью послышался -> предсказано: некоторых, правильно: смех F
  тем более -> предсказано: снег, правильно: людей F
  таким образом -> предсказано: закон, правильно: аккуратно F
  что эта женщина вела там -> предсказано: что, правильно: где F
  что муж женой -> предсказано: так, правильно: спорили F
  этом явлений остановить все свое -> предсказано: горе, правильно: внимание F
  что видел его -> предсказано: душе, правильно: поле F
  когда кабинете -> предсказано: государь, правильно: говорил F
  время как князь андрей -> предсказано: видал, правильно: несвицким F
  ответы эти -> предсказано: были, правильно: вопросы F
  ежели ничего -> предсказано: слыхал, правильно: люблю F
  что произойдет после оставления этой -> предсказано: местности, правильно: позиции F
  рассеянно отвечал -> предсказано: г

In [17]:
# 8. Генерация текста

def generate_text(start_words, max_words=10):
    """Генерирует текст по начальным словам"""
    indices = [word2idx[w] for w in start_words if w in word2idx]
    if not indices:
        print("Слова не найдены в словаре")
        return

    model.eval()
    generated = list(indices)

    with torch.no_grad():
        for _ in range(max_words):
            inp = torch.tensor([generated], dtype=torch.long, device=device)
            lengths = torch.tensor([len(generated)], dtype=torch.long, device=device)
            logits = model(inp, lengths)
            # Берём предсказание на последней позиции
            next_word = logits[0, -1, :].argmax().item()
            generated.append(next_word)

    return ' '.join([idx2word[i] for i in generated])

print("\nГенерация текста:")

start_phrases = [
    ["князь", "андрей"],
    ["наташа", "вдруг"],
    ["русская", "армия"],
    ["пьер", "посмотрел"],
    ["старый", "князь"],
    ["наташа","посмотрела"]
]

for phrase in start_phrases:
    if all(w in word2idx for w in phrase):
        result = generate_text(phrase, max_words=8)
        print(f"  {result}")


Генерация текста:
  князь андрей оживлялся ход особенности особенности особенности дам особенности дам
  наташа вдруг заплакала почувствовала девушку его подругу улыбнулась платье пьеру
  русская армия быть уверены русских войск неприятельских через город вследствие
  пьер посмотрел дивана отцу наташей марьей наташей наташей комнату наташей
  старый князь андрей болконский вечер вечер был назначен день императоров
  наташа посмотрела него небольшие глаза отца такую свой разговор про
